In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "engine" / "pipeline.py").exists())
sys.path.insert(0, str(root / "engine"))
from pipeline import run_case
from IPython.display import display, Image
import ipywidgets as widgets
import pandas as pd
vault = root / "workspace"

In [ ]:
lat = widgets.FloatText(value=49.4875, description="Latitude")
lon = widgets.FloatText(value=8.4660, description="Longitude")
month = widgets.Text(value="2021-01", description="Start month")
button = widgets.Button(description="Run analysis", button_style="primary")
output = widgets.Output()

def run_clicked(_):
    button.disabled = True
    with output:
        output.clear_output()
        try:
            case = run_case(vault, lat.value, lon.value, month.value)
            display(Image(filename=str(case / "monthly_shares.png")))
            display(pd.read_csv(case / "changes.csv"))
            maps = sorted(case.glob("map_*.png"))
            selector = widgets.SelectionSlider(options=[(p.stem[4:], str(p)) for p in maps], description="Month")
            panel = widgets.Output()
            def show(change=None):
                with panel:
                    panel.clear_output(wait=True)
                    display(Image(filename=selector.value, width=512))
            selector.observe(show, names="value")
            display(selector, panel)
            show()
        except Exception as error:
            print(type(error).__name__, str(error))
        finally:
            button.disabled = False

button.on_click(run_clicked)
display(widgets.VBox([lat, lon, month, button, output]))